# Multivariable Calculus

| Difficulty | █████░░░░░ 5/10 |
| :--- | :--- |
| Prerequisites | 02.07 |
| Time | ~65 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/02_Calculus/08_multivariable_calculus.ipynb)

---

## 🎯 Learning Objective

Extend integration to multiple variables with double and triple integrals, and learn how change-of-variables and Fubini's theorem simplify these computations.

---

## 📐 Theory

`02.07` integrated a single-variable function to get a signed area. This notebook extends that
to functions of two or three variables, where the integral measures **volume** (or a
higher-dimensional analogue) instead.

### Double integrals

For $f(x,y)$ over a region $R$, the double integral
$\iint_R f(x,y)\,dA$ is the signed volume between the surface $z=f(x,y)$ and the $xy$-plane,
built the same way as before: partition $R$ into tiny cells of area $\Delta A$, sum
$f(x_i,y_i)\Delta A$ over all cells, and take the limit as the cells shrink to points.

### Fubini's theorem: reducing to iterated 1D integrals

Computing a genuine 2D limit-of-sums is unwieldy. **Fubini's theorem** says that (for
well-behaved $f$) a double integral over a rectangle $[a,b]\times[c,d]$ can be computed as two
ordinary 1D integrals, nested:

$$\iint_{[a,b]\times[c,d]} f(x,y)\,dA = \int_a^b\left(\int_c^d f(x,y)\,dy\right)dx =
\int_c^d\left(\int_a^b f(x,y)\,dx\right)dy$$

Integrate with respect to one variable first (treating the other as fixed, exactly like a
partial derivative in `02.03` but for integration), then integrate the resulting function of
the remaining variable. Fubini's theorem guarantees you get the *same answer* regardless of
which variable you integrate first — this order-independence is the 2D integration analogue of
Clairaut's theorem for mixed partials.

### Triple integrals

The same idea extends to $\iiint_V f(x,y,z)\,dV$ over a 3D region $V$, computing a
4-dimensional "hypervolume" when $f$ isn't constant, or ordinary volume when $f\equiv 1$. Fubini
again lets you peel it apart into three nested 1D integrals.

### Change of variables

Sometimes a region or integrand is awkward in Cartesian coordinates $(x,y)$ but simple in
another coordinate system, like polar $(r,\theta)$. The **change of variables formula** lets
you substitute:

$$\iint_R f(x,y)\,dx\,dy = \iint_{R'} f(x(r,\theta), y(r,\theta))\, |J|\, dr\,d\theta$$

where $|J|$ is the absolute value of the determinant of the Jacobian (recall `02.06`) of the
coordinate transformation — for polar coordinates, $|J| = r$. This $|J|$ factor is a
*stretching correction*: transforming coordinates distorts area/volume locally, and the
Jacobian determinant is exactly the local scaling factor needed to compensate, so the total
integral still measures the same true volume regardless of which coordinates you compute in.

### Why this matters here

Once you reach probability (Module `03`), a joint density $p(x,y)$ over two random variables
must integrate to $1$ over the *whole plane* — a double integral. Recovering the marginal
distribution of just $x$ means integrating $y$ out: $p(x) = \int p(x,y)\,dy$ — a Fubini-style
reduction that appears constantly under the name **marginalization**. Normalizing constants in
Bayesian inference (`03.05`) are exactly this kind of integral, often over many more than two
variables, which is why sampling methods (`03.09`) exist: for high-dimensional integrals,
direct evaluation is often intractable and Monte Carlo estimation becomes the practical tool.

---

In [ ]:
# Setup
import numpy as np
import sympy as sp
import scipy.integrate as spi
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

A 3D surface plot of $f(x,y)$ over a rectangular region, with the volume between the surface
and the plane shaded, makes the double integral's "volume under a surface" meaning concrete.

In [ ]:
# f(x,y) = exp(-(x^2+y^2)) over [-2,2]x[-2,2] -- a bump whose double integral is the volume underneath
f = lambda x, y: np.exp(-(x**2 + y**2))
x = np.linspace(-2, 2, 60)
y = np.linspace(-2, 2, 60)
X, Y = np.meshgrid(x, y)
Z = f(X, Y)

fig = plt.figure(figsize=(12, 5))
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
ax1.plot_surface(X, Y, Z, cmap="viridis", alpha=0.9)
ax1.set_xlabel("x"); ax1.set_ylabel("y"); ax1.set_zlabel("f(x,y)")
ax1.set_title("Surface z=exp(-(x^2+y^2)):\ndouble integral = volume underneath")

def trapz_1d(values, points):
    # Same trapezoidal rule as 02.07's trapezoidal_rule, written for arbitrary sample arrays
    # (avoids relying on a numpy version-specific trapz/trapezoid name)
    widths = np.diff(points)
    return np.sum(widths * (values[:-1] + values[1:]) / 2)

# Riemann-sum-style volume estimate: sum f(x,y) * cell_area over a fine grid (2D generalization
# of the 1D rectangle sums from 02.07)
cell_area = (x[1] - x[0]) * (y[1] - y[0])
riemann_volume = np.sum(Z) * cell_area

# Fubini check: integrate over y first (for each x), then integrate that result over x
inner_integral = np.array([trapz_1d(Z[:, i], y) for i in range(len(x))])  # integrate along y, per x
fubini_volume = trapz_1d(inner_integral, x)  # integrate the result along x -- outer integral

ax2 = fig.add_subplot(1, 2, 2)
ax2.plot(x, inner_integral, color="#4C72B0", lw=2)
ax2.set_xlabel("x"); ax2.set_ylabel(r"$\int f(x,y)\,dy$ (inner integral, for each x)")
ax2.set_title("Fubini's theorem: integrate y out first,\nthen integrate the resulting 1D function")
plt.tight_layout()
plt.show()

exact = np.pi * (1 - np.exp(-4))**1  # approx correction factor for the finite domain vs full plane
print(f"Riemann-sum volume estimate:         {riemann_volume:.5f}")
print(f"Fubini (iterated 1D integrals) volume: {fubini_volume:.5f}")
print(f"(As the domain grows to cover the whole plane, this approaches pi = {np.pi:.5f} exactly --")
print(f" the classic Gaussian integral result used throughout probability, Module 03.)")

## 🐍 Implementation from Scratch

We implement a 2D Riemann sum directly, verify Fubini's theorem by integrating in both orders,
verify the change-of-variables formula on a polar-coordinates example, and cross-check
everything against `scipy.integrate.dblquad`.

In [ ]:
def double_integral_riemann(f, x_range, y_range, n=200):
    """2D Riemann sum: sum f(x,y) over a fine grid, weighted by cell area -- the direct
    generalization of the 1D rectangle sums from 02.07."""
    xs = np.linspace(*x_range, n)
    ys = np.linspace(*y_range, n)
    X, Y = np.meshgrid(xs, ys)
    cell_area = (xs[1] - xs[0]) * (ys[1] - ys[0])
    return np.sum(f(X, Y)) * cell_area

# Fubini check on f(x,y) = x*y^2 over [0,2]x[0,3] -- has a clean closed form to check against
f2 = lambda x, y: x * y**2
riemann_est = double_integral_riemann(f2, (0, 2), (0, 3), n=400)
# NOTE: scipy's dblquad calls func(y, x) -- y first -- so we swap the argument order here
scipy_est, _ = spi.dblquad(lambda y, x: f2(x, y), 0, 2, 0, 3)
x_sym, y_sym = sp.symbols('x y')
exact_sym = sp.integrate(sp.integrate(x_sym * y_sym**2, (y_sym, 0, 3)), (x_sym, 0, 2))

print(f"Riemann sum estimate:        {riemann_est:.5f}")
print(f"scipy.integrate.dblquad:     {scipy_est:.5f}")
print(f"Exact (SymPy, iterated):     {float(exact_sym):.5f}")

# Change of variables: integrate a disk of radius R using polar coordinates, where the
# Jacobian determinant |J| = r makes the substitution correct
R = 2.0
g_polar = lambda r, theta: (r * np.cos(theta))**2 * r  # g(x,y)=x^2, x=r*cos(theta), times |J|=r
polar_est, _ = spi.dblquad(g_polar, 0, 2 * np.pi, 0, R)
# Cross-check directly in Cartesian coordinates using a masked Riemann sum over the same disk
xs = np.linspace(-R, R, 800); ys = np.linspace(-R, R, 800)
Xc, Yc = np.meshgrid(xs, ys)
mask = Xc**2 + Yc**2 <= R**2
cell_area = (xs[1]-xs[0]) * (ys[1]-ys[0])
cartesian_est = np.sum((Xc**2)[mask]) * cell_area

print(f"\nIntegral of x^2 over a disk of radius {R} -- polar (with Jacobian |J|=r): {polar_est:.5f}")
print(f"Same integral estimated directly in Cartesian coordinates:              {cartesian_est:.5f}")
print(f"Agreement confirms the |J|=r correction factor is doing its job.")

## 🤖 Why This Matters for AI

**Marginalization** — recovering $p(x)$ from a joint density $p(x,y)$ by integrating $y$ out —
is exactly the Fubini-style "integrate one variable, holding the rest" operation from this
notebook, and it's everywhere in probabilistic ML: a Gaussian Mixture Model's marginal
likelihood, a VAE's evidence $p(x) = \int p(x|z)p(z)\,dz$, and a Bayesian model's posterior
predictive distribution are all multivariable integrals over latent variables. When these
integrals have no closed form (routine once distributions get complex), Module `03.09`'s
Monte Carlo methods approximate them the same way our Riemann sum approximated a double
integral above: sample points, weight, and average. Below, we compute a 2D Gaussian's
marginal distribution two ways — via the closed-form Gaussian integral, and via a numeric
double-integral marginalization — and confirm they agree.

In [ ]:
from scipy.stats import norm

# A correlated bivariate Gaussian joint density p(x, y) -- a stand-in for a joint distribution
# over, say, a latent variable y and an observation x in a probabilistic model
rho = 0.6  # correlation between x and y
def joint_pdf(x, y):
    return (1 / (2*np.pi*np.sqrt(1-rho**2))) * np.exp(-(x**2 - 2*rho*x*y + y**2) / (2*(1-rho**2)))

# Marginalize y out numerically: p(x) = integral of p(x,y) dy -- exactly the Fubini-style
# "integrate one variable, hold the other fixed" operation from the Theory section
y_grid = np.linspace(-8, 8, 4000)
x_test_points = np.array([-1.5, 0.0, 0.8, 2.0])
marginal_numeric = np.array([trapz_1d(joint_pdf(xv, y_grid), y_grid) for xv in x_test_points])

# Closed form: for THIS joint density, marginalizing out y always recovers a standard normal
# in x -- a known property of the bivariate Gaussian, giving us an exact answer to check against
marginal_exact = norm.pdf(x_test_points, loc=0, scale=1)

print("Marginalizing a joint density by numeric integration (Fubini in action):")
for xv, num, exact in zip(x_test_points, marginal_numeric, marginal_exact):
    print(f"  p(x={xv:+.1f}) via double-integral marginalization: {num:.6f}   exact: {exact:.6f}")
print(f"\nAll match: {np.allclose(marginal_numeric, marginal_exact, atol=1e-4)}")

fig, ax = plt.subplots(figsize=(7, 4.5))
x_fine = np.linspace(-4, 4, 200)
marginal_fine = np.array([trapz_1d(joint_pdf(xv, y_grid), y_grid) for xv in x_fine])
ax.plot(x_fine, marginal_fine, color="#4C72B0", lw=3, label="marginalized numerically (this notebook)")
ax.plot(x_fine, norm.pdf(x_fine), color="#DD8452", lw=1.5, ls="--", label="exact standard normal")
ax.set_xlabel("x"); ax.set_ylabel("p(x)")
ax.set_title("Marginalization: integrating a latent variable out of a joint density")
ax.legend()
plt.show()

## 🏋️ Exercises

### Warm-up
1. Compute $\iint_{[0,1]\times[0,2]} (2x + y)\,dA$ by hand using Fubini's theorem (integrate in
   either order — confirm you get the same answer both ways). Verify with `double_integral_riemann`.

### Practice
2. Use the change-of-variables formula to compute $\iint_D e^{-(x^2+y^2)}\,dA$ where $D$ is the
   disk of radius 3, by converting to polar coordinates (Jacobian $|J|=r$). Verify against a
   masked Cartesian Riemann sum like the one used for the disk example above.

### Challenge
3. Change the correlation `rho` in the AI section's bivariate Gaussian to a few different
   values between $-0.9$ and $0.9$, and confirm the marginal of $x$ is *always* a standard
   normal regardless of `rho` (a special property of Gaussians). Then explain in a sentence
   why this "marginals don't depend on correlation" property would make Gaussians a
   computationally convenient (if sometimes unrealistic) choice of latent prior in a
   probabilistic model.

---

## 📚 Further Reading
- Stewart, *Calculus: Early Transcendentals*, Ch. 15 (Multiple Integrals)
- Bishop, *Pattern Recognition and Machine Learning*, Ch. 2.3 (The Gaussian Distribution)

---

*Next notebook: [Vector Calculus](09_vector_calculus.ipynb)*